# Task 2 Grouped Reward VerdictLabel Submission

This notebook uses the **provided trace verdicts** as hard structure and uses `verdict` as the gold claim label when available.

Pipeline:
1. Train a binary reward model for `useful trace` vs `not useful trace`.
2. For each claim, score all 20 traces.
3. Aggregate scores **inside each provided verdict bucket** (`True`, `False`, `Conflicting`).
4. Predict the claim label from the strongest verdict bucket.
5. Rank traces from the predicted verdict bucket first, then the others.

This is much closer to the actual task than predicting each trace verdict from scratch.


In [ ]:
import json
import os
import re
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.utils.class_weight import compute_class_weight

LANG_CODE = os.environ.get('TASK2_LANG_CODE', 'en')
LANG_NAME = os.environ.get('TASK2_LANG_NAME', 'English')

TASK2_TRAIN_PATH = Path(os.environ.get(
    'TASK2_TRAIN_PATH',
    'clef_2026_checkthat_english_train.json',
))
TASK2_VAL_PATH = Path(os.environ.get(
    'TASK2_VAL_PATH',
    'clef2026_gpt4_o_mini_val.json',
))
TASK2_TEST_PATH = Path(os.environ.get(
    'TASK2_TEST_PATH',
    '/clef_2026_final_english_test.json',
))
OUT_DIR = Path(os.environ.get('TASK2_OUT_DIR', ''))
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['True', 'False', 'Conflicting']
RANDOM_STATE = int(os.environ.get('TASK2_SEED', '42'))
TOTAL_EPOCHS = int(os.environ.get('TASK2_RM_EPOCHS', '6'))
MAX_TRAIN_ROWS = int(os.environ.get('TASK2_MAX_TRAIN_ROWS', '0'))
SUBMISSION_PREFIX = os.environ.get('TASK2_SUBMISSION_PREFIX', 'Task2')

print('Train path:', TASK2_TRAIN_PATH)
print('Val path:', TASK2_VAL_PATH)
print('Test path:', TASK2_TEST_PATH)
print('Output dir:', OUT_DIR)
print('Reward model epochs:', TOTAL_EPOCHS)
print('Max train rows:', MAX_TRAIN_ROWS)

def log_step(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}")

@dataclass
class ClaimExample:
    claim_id: str
    claim_text: str
    label: str | None
    language: str
    split: str
    reasoning_paths: list[str] = field(default_factory=list)
    verdict_list: list[str] = field(default_factory=list)
    evidence_texts: list[str] = field(default_factory=list)


def normalize_label(label):
    if label is None:
        return None
    s = str(label).strip().lower()
    if not s:
        return None
    if s in ('true', 'supported', 'support', 'supports'):
        return 'True'
    if s in ('false', 'refuted', 'refute', 'refutes'):
        return 'False'
    if s in ('conflicting', 'conflict', 'unverifiable', 'unknown'):
        return 'Conflicting'
    return None


def load_task2_claims(claims_path, language='en', split=None):
    path = Path(claims_path)
    if not path.exists():
        return []
    raw = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(raw, dict):
        candidate = raw.get('claims', raw.get('data', raw.get('train', raw.get('val', raw.get('test', [])))))
        if candidate:
            raw = candidate
        elif 'claim' in raw:
            raw = [raw]
        else:
            raw = []
    if not isinstance(raw, list):
        return []

    examples = []
    for i, obj in enumerate(raw):
        if not isinstance(obj, dict):
            continue
        reasoning_paths = obj.get('reasoning_paths', obj.get('Reasoning_traces', obj.get('paths', [])))
        if isinstance(reasoning_paths, str):
            try:
                reasoning_paths = json.loads(reasoning_paths) if reasoning_paths else []
            except json.JSONDecodeError:
                reasoning_paths = []
        verdict_list = obj.get('verdict_list', obj.get('Verdict_list', obj.get('BoN_Verdict_list', [])))
        if isinstance(verdict_list, str):
            try:
                verdict_list = json.loads(verdict_list) if verdict_list else []
            except json.JSONDecodeError:
                verdict_list = []
        evidence_texts = obj.get('evidence_texts', [])
        evidences = obj.get('evidences', [])
        if evidences and not evidence_texts:
            evidence_texts = [e.get('text', e) if isinstance(e, dict) else str(e) for e in evidences]
        examples.append(ClaimExample(
            claim_id=str(obj.get('claim_id', obj.get('id', f'c{i}'))),
            claim_text=str(obj.get('claim_text', obj.get('claim', ''))),
            label=normalize_label(obj.get('verdict', obj.get('label'))),
            language=language,
            split=obj.get('split', split or 'train'),
            reasoning_paths=list(reasoning_paths or []),
            verdict_list=[normalize_label(v) for v in list(verdict_list or [])],
            evidence_texts=list(evidence_texts or []),
        ))
    return examples


def join_evidence(evidence_texts):
    pieces = []
    for text in evidence_texts or []:
        text = str(text).strip()
        if text:
            pieces.append(text)
    return ' '.join(pieces)


NUM_PATTERN = re.compile(r'(?<!\w)(?:\$|€|£)?\d[\d,]*(?:\.\d+)?%?(?!\w)')
YEAR_PATTERN = re.compile(r'(?<!\d)(?:19|20)\d{2}(?!\d)')
MONTH_PATTERN = re.compile(
    r'\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|'
    r'sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)\b',
    re.I,
)


def normalize_numeric_token(token):
    token = str(token).strip().lower()
    token = token.replace(',', '')
    return token


def extract_numeric_tokens(text):
    return [normalize_numeric_token(m.group(0)) for m in NUM_PATTERN.finditer(str(text or ''))]


def extract_year_tokens(text):
    return [m.group(0) for m in YEAR_PATTERN.finditer(str(text or ''))]


def extract_month_tokens(text):
    return [m.group(0).lower()[:3] for m in MONTH_PATTERN.finditer(str(text or ''))]


def clip_token_list(tokens, limit=8):
    return sorted(list(tokens))[:limit]


def build_feature_block(claim, evidence, trace):
    claim_nums = set(extract_numeric_tokens(claim))
    evidence_nums = set(extract_numeric_tokens(evidence))
    trace_nums = set(extract_numeric_tokens(trace))

    claim_years = set(extract_year_tokens(claim))
    evidence_years = set(extract_year_tokens(evidence))
    trace_years = set(extract_year_tokens(trace))

    claim_months = set(extract_month_tokens(claim))
    evidence_months = set(extract_month_tokens(evidence))
    trace_months = set(extract_month_tokens(trace))

    shared_claim_trace_nums = claim_nums & trace_nums
    shared_claim_evidence_nums = claim_nums & evidence_nums
    shared_trace_evidence_nums = trace_nums & evidence_nums
    missing_claim_nums_in_trace = claim_nums - trace_nums

    shared_claim_trace_years = claim_years & trace_years
    shared_trace_evidence_years = trace_years & evidence_years
    shared_claim_trace_months = claim_months & trace_months
    shared_trace_evidence_months = trace_months & evidence_months

    flags = []
    if any('%' in x for x in claim_nums):
        flags.append('CLAIM_HAS_PERCENT')
    if any('%' in x for x in trace_nums):
        flags.append('TRACE_HAS_PERCENT')
    if any(x.startswith(('$', '€', '£')) for x in claim_nums):
        flags.append('CLAIM_HAS_CURRENCY')
    if any(x.startswith(('$', '€', '£')) for x in trace_nums):
        flags.append('TRACE_HAS_CURRENCY')

    parts = [
        f'NUM_COUNT_CLAIM_{len(claim_nums)}',
        f'NUM_COUNT_TRACE_{len(trace_nums)}',
        f'NUM_COUNT_EVIDENCE_{len(evidence_nums)}',
        f'NUM_OVERLAP_CLAIM_TRACE_{len(shared_claim_trace_nums)}',
        f'NUM_OVERLAP_CLAIM_EVIDENCE_{len(shared_claim_evidence_nums)}',
        f'NUM_OVERLAP_TRACE_EVIDENCE_{len(shared_trace_evidence_nums)}',
        f'NUM_MISSING_CLAIM_TRACE_{len(missing_claim_nums_in_trace)}',
        f'YEAR_OVERLAP_CLAIM_TRACE_{len(shared_claim_trace_years)}',
        f'YEAR_OVERLAP_TRACE_EVIDENCE_{len(shared_trace_evidence_years)}',
        f'MONTH_OVERLAP_CLAIM_TRACE_{len(shared_claim_trace_months)}',
        f'MONTH_OVERLAP_TRACE_EVIDENCE_{len(shared_trace_evidence_months)}',
    ]
    parts.extend(flags)
    parts.extend('SHARED_CT_NUM_' + tok for tok in clip_token_list(shared_claim_trace_nums))
    parts.extend('SHARED_TE_NUM_' + tok for tok in clip_token_list(shared_trace_evidence_nums))
    parts.extend('MISSING_CT_NUM_' + tok for tok in clip_token_list(missing_claim_nums_in_trace))
    parts.extend('SHARED_CT_YEAR_' + tok for tok in clip_token_list(shared_claim_trace_years))
    parts.extend('SHARED_TE_YEAR_' + tok for tok in clip_token_list(shared_trace_evidence_years))
    parts.extend('SHARED_CT_MONTH_' + tok for tok in clip_token_list(shared_claim_trace_months))
    return ' '.join(parts)


def build_text(claim, evidence, verdict, trace):
    feature_block = build_feature_block(claim, evidence, trace)
    return (
        'CLAIM: ' + claim +
        '\nEVIDENCE: ' + evidence +
        '\nTRACE_VERDICT: ' + str(verdict) +
        '\nTRACE: ' + trace +
        '\nFEATURES: ' + feature_block
    )


def make_trace_rows(claim_examples, include_label=True):
    rows = []
    for claim_idx, ex in enumerate(claim_examples):
        evidence = join_evidence(getattr(ex, 'evidence_texts', []))
        traces = list(getattr(ex, 'reasoning_paths', []) or [])
        verdicts = list(getattr(ex, 'verdict_list', []) or [])
        for trace_idx, trace in enumerate(traces):
            trace = str(trace or '').strip()
            if not trace:
                continue
            given_verdict = verdicts[trace_idx] if trace_idx < len(verdicts) else None
            given_verdict = normalize_label(given_verdict)
            if given_verdict is None:
                continue
            gold_label = normalize_label(getattr(ex, 'label', None)) if include_label else None
            useful = int(gold_label is not None and given_verdict == gold_label)
            rows.append({
                'claim_idx': claim_idx,
                'claim': ex.claim_text,
                'evidence': evidence,
                'trace': trace,
                'given_verdict': given_verdict,
                'gold_label': gold_label,
                'useful': useful,
            })
    return rows


def get_text(row):
    return build_text(row['claim'], row.get('evidence', ''), row.get('given_verdict', ''), row['trace'])


def title_label(label):
    return normalize_label(label) or 'Conflicting'


Train path: /storage/scratch1/5/sshrestha304/traintestval/clef_2026_checkthat_english_train.json
Val path: /storage/scratch1/5/sshrestha304/traintestval/clef2026_gpt4_o_mini_val.json
Test path: /storage/scratch1/5/sshrestha304/traintestval/clef_2026_final_english_test.json
Output dir: /storage/scratch1/5/sshrestha304/traintestval
Reward model epochs: 6
Max train rows: 0


In [2]:
log_step('Loading train/val/test files')
train_claims = load_task2_claims(TASK2_TRAIN_PATH, language=LANG_CODE, split='train')
val_claims = load_task2_claims(TASK2_VAL_PATH, language=LANG_CODE, split='val')
test_claims = load_task2_claims(TASK2_TEST_PATH, language=LANG_CODE, split='test') if TASK2_TEST_PATH.exists() else []

log_step('Flattening traces')
train_rows = make_trace_rows(train_claims, include_label=True)
val_rows = make_trace_rows(val_claims, include_label=True)
test_rows = make_trace_rows(test_claims, include_label=False)

if MAX_TRAIN_ROWS > 0 and len(train_rows) > MAX_TRAIN_ROWS:
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(len(train_rows), size=MAX_TRAIN_ROWS, replace=False)
    train_rows = [train_rows[i] for i in sorted(idx)]
    log_step(f'Subsampled training rows to {len(train_rows)}')

print('Train claims:', len(train_claims))
print('Val claims:', len(val_claims))
print('Test claims:', len(test_claims))
print('Train traces:', len(train_rows))
print('Val traces:', len(val_rows))
print('Test traces:', len(test_rows))
print('Train useful distribution:', Counter(r['useful'] for r in train_rows))
print('Train verdict distribution:', Counter(r['given_verdict'] for r in train_rows))


[01:54:26] Loading train/val/test files
[01:54:27] Flattening traces
Train claims: 6400
Val claims: 1600
Test claims: 2558
Train traces: 127020
Val traces: 24000
Test traces: 51160
Train useful distribution: Counter({1: 109937, 0: 17083})
Train verdict distribution: Counter({'False': 50618, 'Conflicting': 39508, 'True': 36894})


In [3]:
log_step(f'Building training text for {len(train_rows)} traces')
X_train = [get_text(row) for row in train_rows]
y_train = [row['useful'] for row in train_rows]

reward_model = Pipeline([
    ('features', FeatureUnion([
        ('word_tfidf', TfidfVectorizer(
            analyzer='word',
            ngram_range=(1, 2),
            min_df=3,
            max_features=180000,
            sublinear_tf=True,
            strip_accents='unicode',
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=2,
            max_features=220000,
            sublinear_tf=True,
            strip_accents='unicode',
        )),
    ])),
    ('clf', SGDClassifier(
        loss='log_loss',
        alpha=1.2e-5,
        penalty='l2',
        random_state=RANDOM_STATE,
        average=True,
    )),
])

log_step('Fitting grouped reward model')
t0 = time.time()
feature_step = reward_model.named_steps['features']
clf = reward_model.named_steps['clf']
X_features = feature_step.fit_transform(X_train, y_train)
classes = np.array([0, 1])
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
weight_map = {int(cls): float(w) for cls, w in zip(classes, class_weights)}
sample_weight = np.array([weight_map[int(y)] for y in y_train], dtype=float)
log_step('Class weights: ' + str(weight_map))
for epoch in range(1, TOTAL_EPOCHS + 1):
    epoch_start = time.time()
    clf.partial_fit(X_features, y_train, classes=classes, sample_weight=sample_weight)
    epoch_seconds = time.time() - epoch_start
    elapsed = time.time() - t0
    avg_epoch = elapsed / epoch
    eta = avg_epoch * (TOTAL_EPOCHS - epoch)
    log_step(f'Epoch {epoch}/{TOTAL_EPOCHS} complete | epoch {epoch_seconds:.1f}s | elapsed {elapsed:.1f}s | ETA {eta:.1f}s')

reward_model.steps[0] = ('features', feature_step)
reward_model.steps[1] = ('clf', clf)
log_step(f'Training finished in {time.time() - t0:.1f}s')

try:
    import joblib
    joblib.dump(reward_model, OUT_DIR / 'grouped_reward_model.joblib')
    log_step('Saved checkpoint to ' + str(OUT_DIR / 'grouped_reward_model.joblib'))
except Exception as exc:
    log_step('Skipping checkpoint save: ' + str(exc))


[01:54:28] Building training text for 127020 traces
[01:54:47] Fitting grouped reward model
[01:58:47] Class weights: {0: 3.7177310776795647, 1: 0.5776944977578068}
[01:58:49] Epoch 1/6 complete | epoch 1.8s | elapsed 241.1s | ETA 1205.6s
[01:58:50] Epoch 2/6 complete | epoch 1.9s | elapsed 243.0s | ETA 486.0s
[01:58:52] Epoch 3/6 complete | epoch 1.7s | elapsed 244.7s | ETA 244.7s
[01:58:54] Epoch 4/6 complete | epoch 1.7s | elapsed 246.4s | ETA 123.2s
[01:58:56] Epoch 5/6 complete | epoch 1.7s | elapsed 248.1s | ETA 49.6s
[01:58:57] Epoch 6/6 complete | epoch 1.8s | elapsed 249.9s | ETA 0.0s
[01:58:57] Training finished in 249.9s
[01:59:01] Saved checkpoint to /storage/scratch1/5/sshrestha304/traintestval/grouped_reward_model.joblib


In [4]:
def predict_trace_scores(rows):
    if not rows:
        return []
    texts = [get_text(row) for row in rows]
    probs = reward_model.predict_proba(texts)
    positive_index = list(reward_model.named_steps['clf'].classes_).index(1)
    useful_probs = probs[:, positive_index].astype(float).tolist()
    items = []
    for row, score in zip(rows, useful_probs):
        items.append({
            'claim_idx': row['claim_idx'],
            'claim': row['claim'],
            'trace': row['trace'],
            'given_verdict': row['given_verdict'],
            'gold_label': row.get('gold_label'),
            'score': float(score),
        })
    return items


def bucket_score(verdict_items, topk=5, count_bonus=0.03):
    if not verdict_items:
        return -1.0
    s_items = sorted(verdict_items, key=lambda d: d['score'], reverse=True)
    top = s_items[:topk]
    score_sum = float(sum(item['score'] for item in top))
    return score_sum + count_bonus * len(top)


def rank_claims_grouped(claim_examples, rows, topk_bucket=5, count_bonus=0.03):
    scored = predict_trace_scores(rows)
    grouped = defaultdict(list)
    for item in scored:
        grouped[item['claim_idx']].append(item)

    outputs = []
    for claim_idx, ex in enumerate(claim_examples):
        items = grouped.get(claim_idx, [])
        by_verdict = {label: [] for label in LABELS}
        for item in items:
            by_verdict[item['given_verdict']].append(item)

        verdict_scores = {
            label: bucket_score(by_verdict[label], topk=topk_bucket, count_bonus=count_bonus)
            for label in LABELS
        }
        predicted_label = max(LABELS, key=lambda label: verdict_scores[label])

        pred_bucket = sorted(by_verdict[predicted_label], key=lambda d: d['score'], reverse=True)
        other_labels = [label for label in LABELS if label != predicted_label]
        other_labels = sorted(other_labels, key=lambda label: verdict_scores[label], reverse=True)
        ordered = list(pred_bucket)
        for label in other_labels:
            ordered.extend(sorted(by_verdict[label], key=lambda d: d['score'], reverse=True))

        traces = [item['trace'] for item in ordered]
        verdicts = [item['given_verdict'] for item in ordered]
        scores = [float(item['score']) for item in ordered]
        gold_label = normalize_label(getattr(ex, 'label', None))
        outputs.append({
            'query_id': claim_idx,
            'Claim': ex.claim_text,
            'Label': gold_label,
            'Verdict_BoN': predicted_label if ordered else 'Conflicting',
            'BoN_Verdict_list': verdicts,
            'Reasoning_traces': traces,
            'score_list': scores,
            'gold_label': gold_label,
            'verdict_scores': verdict_scores,
        })
    return outputs


def recall_at_5(output):
    label = output.get('gold_label')
    verdicts = output.get('BoN_Verdict_list', [])
    if not label or not verdicts:
        return 0.0
    total_relevant = sum(1 for v in verdicts if v == label)
    if total_relevant == 0:
        return 0.0
    top5 = verdicts[:5]
    retrieved_relevant = sum(1 for v in top5 if v == label)
    return retrieved_relevant / total_relevant


def mrr_at_5(output):
    label = output.get('gold_label')
    verdicts = output.get('BoN_Verdict_list', [])
    if not label or not verdicts:
        return 0.0
    for idx, v in enumerate(verdicts[:5], start=1):
        if v == label:
            return 1.0 / idx
    return 0.0


def evaluate_outputs(outputs):
    y_true = [o['gold_label'] for o in outputs if o.get('gold_label')]
    y_pred = [o['Verdict_BoN'] for o in outputs if o.get('gold_label')]
    macro = f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0) if y_true else 0.0
    mean_recall5 = float(np.mean([recall_at_5(o) for o in outputs])) if outputs else 0.0
    mean_mrr5 = float(np.mean([mrr_at_5(o) for o in outputs])) if outputs else 0.0
    combo = 0.5 * (macro + mean_recall5)
    return {
        'macro_f1': macro,
        'recall@5': mean_recall5,
        'mrr@5': mean_mrr5,
        'combo': combo,
        'y_true': y_true,
        'y_pred': y_pred,
    }


In [5]:
log_step('Searching aggregation settings on validation')
search_space = []
for topk_bucket in [1, 2, 3, 5]:
    for count_bonus in [0.00, 0.02, 0.03, 0.05]:
        outputs = rank_claims_grouped(val_claims, val_rows, topk_bucket=topk_bucket, count_bonus=count_bonus)
        metrics = evaluate_outputs(outputs)
        search_space.append({
            'topk_bucket': topk_bucket,
            'count_bonus': count_bonus,
            'outputs': outputs,
            **metrics,
        })
        log_step(
            f"topk_bucket={topk_bucket} count_bonus={count_bonus:.2f} | "
            f"Macro-F1={metrics['macro_f1']:.4f} Recall@5={metrics['recall@5']:.4f} Combo={metrics['combo']:.4f}"
        )

best_run = max(search_space, key=lambda d: d['combo'])
best_outputs = best_run['outputs']

print('\n=== Best Validation Metrics ===')
print('topk_bucket:', best_run['topk_bucket'])
print('count_bonus:', best_run['count_bonus'])
print(f"Macro-F1: {best_run['macro_f1']:.4f}")
print(f"Recall@5: {best_run['recall@5']:.4f}")
print(f"MRR@5:    {best_run['mrr@5']:.4f}")
print(f"Combo:    {best_run['combo']:.4f}")
print('\nClassification report:')
print(classification_report(best_run['y_true'], best_run['y_pred'], labels=LABELS, zero_division=0))

for sample in best_outputs[:3]:
    print('\nSample claim:', sample['Claim'][:200])
    print('Gold:', sample['gold_label'], '| BoN:', sample['Verdict_BoN'])
    print('Verdict scores:', sample['verdict_scores'])
    print('Top-3 verdicts:', sample['BoN_Verdict_list'][:3])
    print('Top-3 scores:', sample['score_list'][:3])


[01:59:01] Searching aggregation settings on validation
[01:59:56] topk_bucket=1 count_bonus=0.00 | Macro-F1=0.8570 Recall@5=0.3411 Combo=0.5991
[02:00:51] topk_bucket=1 count_bonus=0.02 | Macro-F1=0.8570 Recall@5=0.3411 Combo=0.5991
[02:01:46] topk_bucket=1 count_bonus=0.03 | Macro-F1=0.8570 Recall@5=0.3411 Combo=0.5991
[02:02:38] topk_bucket=1 count_bonus=0.05 | Macro-F1=0.8570 Recall@5=0.3411 Combo=0.5991
[02:03:31] topk_bucket=2 count_bonus=0.00 | Macro-F1=0.8796 Recall@5=0.3451 Combo=0.6124
[02:04:27] topk_bucket=2 count_bonus=0.02 | Macro-F1=0.8796 Recall@5=0.3451 Combo=0.6124
[02:05:21] topk_bucket=2 count_bonus=0.03 | Macro-F1=0.8796 Recall@5=0.3451 Combo=0.6124
[02:06:13] topk_bucket=2 count_bonus=0.05 | Macro-F1=0.8796 Recall@5=0.3451 Combo=0.6124
[02:07:07] topk_bucket=3 count_bonus=0.00 | Macro-F1=0.9001 Recall@5=0.3525 Combo=0.6263
[02:08:07] topk_bucket=3 count_bonus=0.02 | Macro-F1=0.9009 Recall@5=0.3526 Combo=0.6268
[02:09:00] topk_bucket=3 count_bonus=0.03 | Macro-F1=0

In [6]:
if test_claims and test_rows:
    log_step('Scoring test set with best grouped aggregation')
    test_outputs = rank_claims_grouped(
        test_claims,
        test_rows,
        topk_bucket=best_run['topk_bucket'],
        count_bonus=best_run['count_bonus'],
    )

    submission = []
    for out in test_outputs:
        submission.append({
            'query_id': out['query_id'],
            'Claim': out['Claim'],
            'Verdict_BoN': out['Verdict_BoN'],
            'BoN_Verdict_list': out['BoN_Verdict_list'],
            'Reasoning_traces': out['Reasoning_traces'],
            'score_list': out['score_list'],
        })

    json_path = OUT_DIR / f'{SUBMISSION_PREFIX}_Numerical_claims_{LANG_NAME}.json'
    zip_path = OUT_DIR / f'{SUBMISSION_PREFIX}_Numerical_claims_{LANG_NAME}.zip'
    json_path.write_text(json.dumps(submission, ensure_ascii=False, indent=2), encoding='utf-8')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname=json_path.name)

    debug_path = OUT_DIR / 'debug_grouped_reward_val_outputs.json'
    debug_path.write_text(json.dumps(best_outputs, ensure_ascii=False, indent=2), encoding='utf-8')

    print('Wrote submission JSON:', json_path)
    print('Wrote submission ZIP:', zip_path)
    print('Wrote validation debug JSON:', debug_path)
else:
    print('No test rows available; skipped submission export.')


[02:13:23] Scoring test set with best grouped aggregation
Wrote submission JSON: /storage/scratch1/5/sshrestha304/traintestval/Task2_Numerical_claims_English.json
Wrote submission ZIP: /storage/scratch1/5/sshrestha304/traintestval/Task2_Numerical_claims_English.zip
Wrote validation debug JSON: /storage/scratch1/5/sshrestha304/traintestval/debug_grouped_reward_val_outputs.json
